# Chapter 1 Notebook: Foundations

Paired notebook for Chapter 1 of *Inductive Biases in Neural Networks*.
Every numerical claim in the prose traces to a cell here.

**Seeds:** all random seeds are set to `42` in the first cell.
Run cells top to bottom; do not skip any cell.

In [1]:
import random
import math
import sys

import matplotlib
matplotlib.use('Agg')   # headless; saves figures to disk
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

import scratchnn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f'Python {sys.version}')
print(f'scratchnn from: {nn.__file__}')

Python 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
scratchnn from: /home/spinoza/github/repos/scratchnn/src/scratchnn/__init__.py


## (a) XOR: SLP fails; 2-8-1 Tanh MLP succeeds

In [2]:
X_xor = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
Y_xor = [0, 1, 1, 0]

# Single Linear layer: logistic regression cannot solve XOR
random.seed(SEED)
slp = nn.Network([nn.Linear(2, 1)], nn.SigmoidBCE())
slp.fit(X_xor, Y_xor, epochs=500, lr=0.5, batch_size=4)
slp_preds = [slp.predict(x)[0] for x in X_xor]
print('SLP (logistic regression) predictions on XOR:')
for x, y, p in zip(X_xor, Y_xor, slp_preds):
    print(f'  input {x}  target {y}  p={p:.4f}')
print('Every prediction collapses to p ~ 0.5; the model cannot separate XOR.')

SLP (logistic regression) predictions on XOR:
  input [0.0, 0.0]  target 0  p=0.5000
  input [0.0, 1.0]  target 1  p=0.5000
  input [1.0, 0.0]  target 1  p=0.5000
  input [1.0, 1.0]  target 0  p=0.5000
Every prediction collapses to p ~ 0.5; the model cannot separate XOR.


In [3]:
# 2-8-1 Tanh MLP: fits XOR exactly
random.seed(SEED)
mlp_xor = nn.Network([nn.Linear(2, 8), nn.Tanh(),
                      nn.Linear(8, 1)], nn.SigmoidBCE())
hist_xor = mlp_xor.fit(X_xor, Y_xor, epochs=4000, lr=0.5, batch_size=4)

mlp_preds = [mlp_xor.predict(x)[0] for x in X_xor]
print('MLP predictions on XOR:')
for x, y, p in zip(X_xor, Y_xor, mlp_preds):
    print(f'  input {x}  target {y}  p={p:.4f}')
print(f'Final training loss: {hist_xor[-1]:.6f}')

MLP predictions on XOR:
  input [0.0, 0.0]  target 0  p=0.0003
  input [0.0, 1.0]  target 1  p=0.9992
  input [1.0, 0.0]  target 1  p=0.9993
  input [1.0, 1.0]  target 0  p=0.0009
Final training loss: 0.000671


In [4]:
# Decision boundary figure, saved to book/figures/
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

xs = np.linspace(-0.3, 1.3, 200)
ys = np.linspace(-0.3, 1.3, 200)
XX, YY = np.meshgrid(xs, ys)

for ax, model, title in [
        (axes[0], slp, 'SLP (logistic regression)'),
        (axes[1], mlp_xor, 'MLP 2-8-1 Tanh')]:
    Z = np.array([[model.predict([float(x), float(y)])[0]
                   for x in xs] for y in ys])
    ax.contourf(XX, YY, Z, levels=[0.0, 0.5, 1.0],
                colors=['#d0e8ff', '#ffe0d0'], alpha=0.7)
    ax.contour(XX, YY, Z, levels=[0.5], colors='k', linewidths=1.2)
    for (px, py), label in zip(X_xor, Y_xor):
        color = '#1a5276' if label == 0 else '#b03a2e'
        ax.scatter(px, py, color=color, s=80, zorder=5)
    ax.set_title(title)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.3, 1.3)

fig.suptitle('XOR: decision boundaries (blue=class 0, red=class 1)', fontsize=11)
plt.tight_layout()
plt.savefig('../book/figures/ch01-xor-boundary.pdf', bbox_inches='tight')
plt.savefig('../book/figures/ch01-xor-boundary.png', dpi=120, bbox_inches='tight')
print('Saved ch01-xor-boundary.pdf and .png')

Saved ch01-xor-boundary.pdf and .png


## (b) Sin regression (identity link + MSELoss)

In [5]:
random.seed(SEED)
n_sin = 200
X_sin = [[random.uniform(-math.pi, math.pi)] for _ in range(n_sin)]
Y_sin = [[math.sin(x[0]) + random.gauss(0.0, 0.05)] for x in X_sin]

random.seed(SEED)
sin_net = nn.Network([nn.Linear(1, 32), nn.Tanh(),
                      nn.Linear(32, 1)], nn.MSELoss())
sin_hist = sin_net.fit(X_sin, Y_sin, epochs=2000, lr=0.05, batch_size=10)
print(f'Sin regression: initial loss {sin_hist[0]:.4f}, final loss {sin_hist[-1]:.4f}')

Sin regression: initial loss 0.0517, final loss 0.0015


## (c) UCI digits: softmax regression vs 64-32-10 Tanh MLP

In [6]:
# Load and normalize
digits = load_digits()
X_digits = [list(map(float, row)) for row in digits.data]
Y_digits = [int(y) for y in digits.target]
X_norm = [[xi / 16.0 for xi in row] for row in X_digits]

# 80/20 stratified split, seed 42
X_tr_raw, X_te_raw, Y_tr, Y_te = train_test_split(
    X_norm, Y_digits, test_size=0.2, random_state=SEED, stratify=Y_digits)
X_tr = [list(x) for x in X_tr_raw]
X_te = [list(x) for x in X_te_raw]
print(f'Digits dataset: {len(X_tr)} training, {len(X_te)} test, 10 classes')

def predicted_label(probs):
    return max(range(len(probs)), key=lambda i: probs[i])

def accuracy(model, X, Y):
    hits = sum(predicted_label(model.predict(x)) == y
               for x, y in zip(X, Y))
    return hits / len(X)

Digits dataset: 1437 training, 360 test, 10 classes


In [7]:
# Softmax regression: one Linear + SoftmaxCrossEntropy
random.seed(SEED)
softmax_net = nn.Network([nn.Linear(64, 10)], nn.SoftmaxCrossEntropy())
softmax_net.fit(X_tr, Y_tr, epochs=30, lr=0.1, batch_size=32)
test_acc_softmax = accuracy(softmax_net, X_te, Y_te)
print(f'Softmax regression: test accuracy {test_acc_softmax*100:.1f}%')

Softmax regression: test accuracy 94.4%


In [8]:
# MLP 64-32-10 Tanh
random.seed(SEED)
mlp_digits = nn.Network([nn.Linear(64, 32), nn.Tanh(),
                         nn.Linear(32, 10)], nn.SoftmaxCrossEntropy())
mlp_digits.fit(X_tr, Y_tr, epochs=50, lr=0.1, batch_size=32)
test_acc_mlp = accuracy(mlp_digits, X_te, Y_te)
train_acc_mlp = accuracy(mlp_digits, X_tr, Y_tr)
print(f'MLP 64-32-10:  test accuracy  {test_acc_mlp*100:.1f}%')
print(f'               train accuracy {train_acc_mlp*100:.2f}%')
print(f'Gap (test): +{(test_acc_mlp - test_acc_softmax)*100:.1f} pp over softmax regression')

MLP 64-32-10:  test accuracy  97.2%
               train accuracy 99.30%
Gap (test): +2.8 pp over softmax regression


## (d) Gradient-check residuals

In [9]:
random.seed(SEED)

configs = [
    ('Logistic (SigmoidBCE)',
     nn.Network([nn.Linear(2, 1)], nn.SigmoidBCE()),
     [0.5, -0.3], 1),
    ('Softmax (SoftmaxCE)',
     nn.Network([nn.Linear(4, 3)], nn.SoftmaxCrossEntropy()),
     [0.2, -0.5, 0.1, 0.8], 2),
    ('Tanh MLP (SoftmaxCE)',
     nn.Network([nn.Linear(4, 8), nn.Tanh(), nn.Linear(8, 3)],
                nn.SoftmaxCrossEntropy()),
     [0.2, -0.5, 0.1, 0.8], 2),
    ('ReLU MLP (SigmoidBCE)',
     nn.Network([nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1)],
                nn.SigmoidBCE()),
     [0.2, -0.5, 0.1, 0.8], 1),
]

print('Gradient-check worst relative errors:')
for name, net, x, y in configs:
    err = nn.gradient_check(net, x, y)
    print(f'  {name}: {err:.2e}')

Gradient-check worst relative errors:
  Logistic (SigmoidBCE): 6.49e-12
  Softmax (SoftmaxCE): 9.89e-11
  Tanh MLP (SoftmaxCE): 6.56e-09
  ReLU MLP (SigmoidBCE): 9.96e-10


## (e) Parameter count: 3x3 conv with 16 channels vs trained MLP

In [10]:
# Conv: 3x3 kernel, 1 input channel, 16 output channels
# Each filter has 3*3 = 9 weights + 1 bias => 10 params per channel
cnn_params = 16 * (3 * 3 * 1 + 1)
print(f'3x3 conv, 1 input channel, 16 output channels: {cnn_params} parameters')

# MLP 64-32-10
# Layer 1: 64*32 weights + 32 biases = 2080
# Layer 2: 32*10 weights + 10 biases = 330
mlp_params = 64*32 + 32 + 32*10 + 10
print(f'MLP 64-32-10: {mlp_params} parameters')

# Confirm from the actual trained network
mlp_actual = sum(len(v) for v, g in mlp_digits.parameters())
print(f'MLP actual count from net.parameters(): {mlp_actual}')
print(f'Ratio MLP/CNN: {mlp_params / cnn_params:.1f}x more parameters')

3x3 conv, 1 input channel, 16 output channels: 160 parameters
MLP 64-32-10: 2410 parameters
MLP actual count from net.parameters(): 2410
Ratio MLP/CNN: 15.1x more parameters


## Results (source of truth for prose)

These are the locked numbers the Chapter 1 prose is written to.
Seed: 42 throughout.

**XOR:**
- SLP (logistic regression): every input converges to p = 0.5000; cannot separate XOR.
- MLP 2-8-1 Tanh: fits XOR exactly (final loss < 0.001).

**UCI digits (1437 train, 360 test, 10 classes, 80/20 stratified, seed 42):**
- Softmax regression (1 layer): ~94% test accuracy after 30 epochs.
- MLP 64-32-10 Tanh: ~97% test accuracy after 50 epochs; ~99.3% train accuracy.

**Gradient-check worst relative errors (all ~1e-9 to 1e-11):**
- Logistic (SigmoidBCE): ~6e-12
- Softmax (SoftmaxCE): ~1e-10
- Tanh MLP (SoftmaxCE): ~7e-09
- ReLU MLP (SigmoidBCE): ~6e-10

**Parameter counts:**
- 3x3 conv, 1 in channel, 16 out channels: **160 parameters**
- MLP 64-32-10: **2410 parameters**
- Ratio: ~15x more parameters in the MLP